In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from skopt import BayesSearchCV

# ================================
# CARREGAR E PREPARAR OS DADOS 
# =================================
df = pd.read_csv('titanic completo.csv')

# Mapear variáveis de texto para números
df['Sex'] = df['sex'].map({'female': 1, 'male': 0})
df['Embarked'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2}).fillna(0)
df['fare'] = df['fare'].fillna(df['fare'].median())

# Seleção de atributos principais
colunas = ['pclass', 'Sex', 'age', 'sibsp', 'parch', 'fare', 'Embarked']
X = df[colunas].copy()
y = df['survived'].copy()

# Divisão de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==================================================
# IMPUTAÇÃO DE DADOS AUSENTES (KNN vs MissForest)
# ==================================================
# KNN Imputer 
knn = KNNImputer(n_neighbors=5)
X_train_knn = pd.DataFrame(knn.fit_transform(X_train), columns=colunas)
X_test_knn = pd.DataFrame(knn.transform(X_test), columns=colunas)

#  MissForest
rf_estimator = RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1)
missforest = IterativeImputer(estimator=rf_estimator, max_iter=5, random_state=42)
X_train_mf = pd.DataFrame(missforest.fit_transform(X_train), columns=colunas)
X_test_mf = pd.DataFrame(missforest.transform(X_test), columns=colunas)

print("--- COMPARAÇÃO DA IDADE (IDADE ORIGINAL VS IMPUTADA) ---")
print("Original   -> Média:", round(X_train['age'].mean(), 2), "| Desvio:", round(X_train['age'].std(), 2))
print("KNN        -> Média:", round(X_train_knn['age'].mean(), 2), "| Desvio:", round(X_train_knn['age'].std(), 2))
print("MissForest -> Média:", round(X_train_mf['age'].mean(), 2), "| Desvio:", round(X_train_mf['age'].std(), 2))

# =============================
#  BALANCEAMENTO DE CLASSES 
# =============================
# SMOTE
smote = SMOTE(random_state=42)
X_tr_smote, y_tr_smote = smote.fit_resample(X_train_mf, y_train)

# Random Under Sampling
rus = RandomUnderSampler(random_state=42)
X_tr_rus, y_tr_rus = rus.fit_resample(X_train_mf, y_train)

# ===================================================
# OTIMIZAÇÃO DE HIPERPARÂMETROS (ÁRVORE DE DECISÃO)
# ===================================================
dt_params = {
    'max_depth': [3, 4, 5, 6, 8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Random Search
t0 = time.time()
rs_dt = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), dt_params, n_iter=15, cv=5, scoring='f1', random_state=42)
rs_dt.fit(X_tr_smote, y_tr_smote)
tempo_rs_dt = time.time() - t0

# Otimização Bayesiana 
t0 = time.time()
bayes_dt = BayesSearchCV(DecisionTreeClassifier(random_state=42), dt_params, n_iter=15, cv=5, scoring='f1', random_state=42)
bayes_dt.fit(X_tr_smote, y_tr_smote)
tempo_bayes_dt = time.time() - t0

# ===================================================
# 5. OTIMIZAÇÃO DE HIPERPARÂMETROS (RANDOM FOREST)
# ===================================================
rf_params = {
    'n_estimators': [50, 100, 150],
    'max_depth': [4, 6, 8],
    'min_samples_split': [2, 5, 10]
}

# Random Search
t0 = time.time()
rs_rf = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_params, n_iter=10, cv=5, scoring='f1', random_state=42)
rs_rf.fit(X_tr_smote, y_tr_smote)
tempo_rs_rf = time.time() - t0

# Otimização Bayesiana 
t0 = time.time()
bayes_rf = BayesSearchCV(RandomForestClassifier(random_state=42), rf_params, n_iter=10, cv=5, scoring='f1', random_state=42)
bayes_rf.fit(X_tr_smote, y_tr_smote)
tempo_bayes_rf = time.time() - t0

melhor_arvore = bayes_dt.best_estimator_
regras = export_text(melhor_arvore, feature_names=colunas, max_depth=3)
print("\n--- REGRAS DA ÁRVORE DE DECISÃO ---")
print(regras)

--- COMPARAÇÃO DA IDADE (IDADE ORIGINAL VS IMPUTADA) ---
Original   -> Média: 29.6 | Desvio: 14.36
KNN        -> Média: 29.81 | Desvio: 13.39
MissForest -> Média: 29.46 | Desvio: 13.47

--- REGRAS DA ÁRVORE DE DECISÃO ---
|--- Sex <= 0.00
|   |--- age <= 8.52
|   |   |--- sibsp <= 2.50
|   |   |   |--- fare <= 15.57
|   |   |   |   |--- class: 1
|   |   |   |--- fare >  15.57
|   |   |   |   |--- class: 1
|   |   |--- sibsp >  2.50
|   |   |   |--- parch <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- parch >  1.50
|   |   |   |   |--- class: 0
|   |--- age >  8.52
|   |   |--- pclass <= 1.98
|   |   |   |--- age <= 36.99
|   |   |   |   |--- class: 1
|   |   |   |--- age >  36.99
|   |   |   |   |--- class: 0
|   |   |--- pclass >  1.98
|   |   |   |--- age <= 32.14
|   |   |   |   |--- class: 0
|   |   |   |--- age >  32.14
|   |   |   |   |--- class: 0
|--- Sex >  0.00
|   |--- pclass <= 3.00
|   |   |--- fare <= 26.02
|   |   |   |--- pclass <= 2.11
|   |   |   |   |--- clas